# MI-GAN Distillation Training
GPU Runtime required: **Runtime → Change runtime type → T4/A100**

In [ ]:
# 1. Clone repo + install deps
!git clone https://github.com/nghyane/Typoon.git
%cd Typoon/scripts/distill
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q onnxruntime-gpu onnx Pillow numpy tensorboard tqdm

In [ ]:
# 2. Download training data from GitHub release
!mkdir -p ../../data/training ../../models

import urllib.request, zipfile, os

BASE = 'https://github.com/nghyane/Typoon/releases/download/training-data-v1'
files = ['training_manga.zip', 'training_manhwa.zip', 'lama_fp32.zip']

for f in files:
    path = f'/tmp/{f}'
    if not os.path.exists(path):
        print(f'Downloading {f}...')
        urllib.request.urlretrieve(f'{BASE}/{f}', path)
    print(f'Extracting {f}...')
    dest = '../../models/' if 'lama' in f else '../../data/training/'
    with zipfile.ZipFile(path, 'r') as z:
        z.extractall(dest)

print('Done!')
!find ../../data/training -type f | wc -l
!ls -lh ../../models/lama_fp32.onnx

In [ ]:
# 3. Verify GPU
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}, {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

# Auto batch size
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
if vram_gb >= 30:    BATCH = 32
elif vram_gb >= 14:  BATCH = 16
elif vram_gb >= 10:  BATCH = 8
else:                BATCH = 4
print(f'Auto batch size: {BATCH}')

In [ ]:
# 4. Generate teacher cache (LaMa on GPU, ~30-60 min)
!python generate_teacher_cache.py \
  --data_dir ../../data/training \
  --output_dir ../../data/teacher_cache \
  --lama_model ../../models/lama_fp32.onnx \
  --masks_per_image 5 \
  --batch_size 4 \
  --random_crop

In [ ]:
# 5. Train MI-GAN from cache
!python train.py \
  --data_dir ../../data/training \
  --teacher_cache ../../data/teacher_cache \
  --batch_size {BATCH} \
  --num_workers 2 \
  --use_amp \
  --total_kimg 500 \
  --lr 1e-3 \
  --kd_weight 2.0 \
  --r1_gamma 10.0 \
  --log_every 50 \
  --vis_every 200 \
  --save_every 2000 \
  --output_dir ../../runs/migan_distill

In [ ]:
# 6. Export to ONNX
!python export_onnx.py \
  --checkpoint ../../runs/migan_distill/checkpoints/best.pt \
  --output ../../runs/migan_inpaint.onnx

In [ ]:
# 7. Download results
from google.colab import files
files.download('../../runs/migan_inpaint.onnx')
files.download('../../runs/migan_distill/checkpoints/best.pt')